In [ ]:
# ==============================================================================
# MODELAGEM DIRETA FDTD CLASSICA E VISUALIZACAO DE CAMPO DE ONDA
# Arquitetura: Deepwave (Referencia da Baseline do Dr. Bruno)
# ==============================================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from typing import Tuple
import deepwave
from deepwave import scalar

def configurar_dispositivo_hardware() -> torch.device:
    """
    Avalia a disponibilidade de hardware e atribui o dispositivo de processamento ideal.
    Prioriza arquiteturas CUDA para as operacoes de tensores do Deepwave.

    Retorna:
        torch.device: O dispositivo PyTorch selecionado ('cuda' ou 'cpu').
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Sistema] Hardware mapeado para: {device}")
    return device

def construir_modelo_velocidade_duas_camadas(
    ny: int, 
    nz: int, 
    vel_camada_01: float, 
    vel_camada_02: float, 
    profundidade_interface_z: int, 
    device: torch.device
) -> torch.Tensor:
    """
    Constroi um tensor de modelo de velocidade acustica 2D discreto com duas camadas horizontais.

    Args:
        ny (int): Numero de pontos do grid na direcao horizontal (Y).
        nz (int): Numero de pontos do grid na direcao vertical de profundidade (Z).
        vel_camada_01 (float): Velocidade acustica da camada superior (m/s).
        vel_camada_02 (float): Velocidade acustica da camada inferior (m/s).
        profundidade_interface_z (int): Indice do eixo Z onde a segunda camada comeca.
        device (torch.device): O dispositivo de hardware para alocacao do tensor.

    Retorna:
        torch.Tensor: Um tensor de velocidade 2D de formato (ny, nz) representando o meio geologico.
    """
    model_np = (np.ones((nz, ny)) * vel_camada_01).astype(np.float32)
    model_np[profundidade_interface_z:, :] = vel_camada_02
    
    # A transposicao (.T) e necessaria porque o Deepwave espera o formato [ny, nz]
    model_tensor = torch.tensor(model_np, dtype=torch.float32, device=device).T
    return model_tensor

def configurar_aquisicao_sismica(
    device: torch.device,
    nt: int,
    dt: float,
    freq_pico: float,
    ny: int
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Configura a wavelet da fonte, a localizacao da fonte e o arranjo de receptores para a simulacao.

    Args:
        device (torch.device): O dispositivo de hardware para alocacao do tensor.
        nt (int): Numero total de passos de tempo.
        dt (float): Intervalo de amostragem de tempo (segundos).
        freq_pico (float): Frequencia de pico da wavelet de Ricker (Hz).
        ny (int): Tamanho do grid horizontal, usado para centralizar a fonte.

    Retorna:
        Tuple[torch.Tensor, torch.Tensor, torch.Tensor]: 
            - Tensor de amplitudes da fonte (Wavelet de Ricker).
            - Tensor de localizacoes da fonte [num_tiros, num_fontes, 2].
            - Tensor de localizacoes dos receptores [num_tiros, num_receptores, 2].
    """
    tempo_pico = 1.0 / freq_pico
    num_tiros = 1
    
    # Configuracao da Fonte
    profundidade_fonte = 10
    localizacoes_fonte = torch.zeros(num_tiros, 1, 2, dtype=torch.long, device=device)
    localizacoes_fonte[..., 1] = profundidade_fonte
    localizacoes_fonte[:, 0, 0] = ny // 2  # Tiro centralizado
    
    # Configuracao dos Receptores (Arranjo cobrindo a superficie)
    num_receptores = 76
    profundidade_receptor = 10
    localizacoes_receptores = torch.zeros(num_tiros, num_receptores, 2, dtype=torch.long, device=device)
    localizacoes_receptores[..., 1] = profundidade_receptor
    localizacoes_receptores[:, :, 0] = torch.arange(num_receptores) + 10
    
    # Geracao da Wavelet da Fonte (Ricker)
    amplitudes_fonte = (
        deepwave.wavelets.ricker(freq_pico, nt, dt, tempo_pico)
        .repeat(num_tiros, 1, 1)
        .to(device)
    )
    
    return amplitudes_fonte, localizacoes_fonte, localizacoes_receptores

class CallbackAvancoTempo:
    """
    Classe de callback invocavel usada pelo Deepwave para capturar snapshots (quadros) 
    do campo de onda em propagacao em passos de tempo predefinidos.
    """
    def __init__(self, indice_tiro: int, tensor_buffer: torch.Tensor):
        """
        Inicializa o mecanismo de captura.
        
        Args:
            indice_tiro (int): O indice do tiro especifico a ser monitorado.
            tensor_buffer (torch.Tensor): Tensor pre-alocado na CPU para armazenar os snapshots.
        """
        self.indice_tiro = indice_tiro
        self.passo_atual = 0
        self.tensor_buffer = tensor_buffer

    def __call__(self, estado: deepwave.common.State) -> None:
        """
        Extrai o estado do campo de onda no passo de tempo interno atual.
        
        Args:
            estado (deepwave.common.State): O objeto de estado interno do motor de calculo.
        """
        self.tensor_buffer[self.passo_atual] = estado.get_wavefield("wavefield_0")[self.indice_tiro].cpu().clone()
        self.passo_atual += 1

def executar_simulacao_direta_deepwave(
    modelo_velocidade: torch.Tensor,
    dx: float,
    dt: float,
    amplitudes_fonte: torch.Tensor,
    localizacoes_fonte: torch.Tensor,
    localizacoes_receptores: torch.Tensor,
    freq_pico: float,
    snapshots_avanco: torch.Tensor,
    frequencia_callback: int = 1
) -> torch.Tensor:
    """
    Executa a operacao de modelagem direta FDTD classica usando o motor Deepwave.

    Args:
        modelo_velocidade (torch.Tensor): O modelo fisico 2D de velocidades.
        dx (float): Espacamento do grid em metros (assume dx=dz isotropico).
        dt (float): Intervalo de passo de tempo em segundos.
        amplitudes_fonte (torch.Tensor): A energia injetada ao longo do tempo.
        localizacoes_fonte (torch.Tensor): Coordenadas de grid das fontes.
        localizacoes_receptores (torch.Tensor): Coordenadas de grid dos receptores.
        freq_pico (float): Frequencia de pico para ajuste das Camadas Perfeitamente Casadas (PML).
        snapshots_avanco (torch.Tensor): Tensor de buffer para capturar quadros do campo de onda.
        frequencia_callback (int): Intervalo de passos para acionar a captura de quadros.

    Retorna:
        torch.Tensor: Os sismogramas gravados nas posicoes dos receptores.
    """
    print("[Simulacao] Iniciando propagacao FDTD...")
    
    # Garante que o rastreamento de gradiente esteja ativado para calculos FWI/Adjuntos no futuro
    vmodel = modelo_velocidade.clone().requires_grad_(True)
    
    # Executa a propagacao da onda acustica escalar
    out = scalar(
        vmodel, 
        dx, 
        dt,
        max_vel=2500.0,
        source_amplitudes=amplitudes_fonte,
        source_locations=localizacoes_fonte,
        receiver_locations=localizacoes_receptores,
        accuracy=8,
        pml_freq=freq_pico,
        pml_width=[30, 30, 30, 30],
        forward_callback=CallbackAvancoTempo(indice_tiro=0, tensor_buffer=snapshots_avanco),
        callback_frequency=frequencia_callback
    )
    
    print("[Simulacao] Propagacao FDTD concluida com sucesso.")
    return out[-1]

def renderizar_animacao_campo_onda(
    snapshots: torch.Tensor, 
    dt: float, 
    frequencia_callback: int
) -> HTML:
    """
    Compila os quadros capturados do campo de onda em um formato de video HTML5 reproduzivel.

    Args:
        snapshots (torch.Tensor): Tensor contendo cortes temporais do campo de onda.
        dt (float): Tempo fisico decorrido por passo de simulacao.
        frequencia_callback (int): Numero de passos entre cada quadro registrado.

    Retorna:
        IPython.display.HTML: O objeto de animacao interativa.
    """
    print("[Visualizacao] Compilando animacao do campo de onda...")
    vmax = torch.quantile(snapshots, 0.99).item()
    
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(snapshots[1].T, cmap="seismic", vmax=vmax, vmin=-vmax, animated=True)
    
    ax.set_title("Propagacao do Campo de Onda (t = 0.000 s)", fontsize=12)
    ax.set_xlabel("Distancia (nos do grid)", fontsize=10)
    ax.set_ylabel("Profundidade (nos do grid)", fontsize=10)
    plt.close(fig) 

    def atualizar(frame: int) -> Tuple:
        im.set_data(snapshots[frame].T)
        t_fisico = frame * frequencia_callback * dt
        ax.set_title(f"Propagacao FDTD Classica (t = {t_fisico:.3f} s)")
        return (im,)

    ani = animation.FuncAnimation(
        fig, 
        atualizar, 
        frames=len(snapshots), 
        interval=25, 
        blit=True
    )
    
    return HTML(ani.to_jshtml())

# ------------------------------------------------------------------------------
# PIPELINE DE EXECUCAO PRINCIPAL
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    
    # Configuracao de Parametros Globais
    GRID_NY, GRID_NZ = 96, 48
    GRID_DX, GRID_DT = 10.0, 0.004
    PASSOS_NT = 480
    FREQ_PICO = 15.0
    
    # 1. Configuracao do Sistema
    dispositivo_atual = configurar_dispositivo_hardware()
    
    # 2. Criacao do Modelo
    modelo_velocidade_real = construir_modelo_velocidade_duas_camadas(
        ny=GRID_NY, 
        nz=GRID_NZ, 
        vel_camada_01=1500.0, 
        vel_camada_02=2000.0, 
        profundidade_interface_z=35, 
        device=dispositivo_atual
    )
    
    # 3. Configuracao de Aquisicao
    amps_fonte, locs_fonte, locs_receptores = configurar_aquisicao_sismica(
        device=dispositivo_atual,
        nt=PASSOS_NT,
        dt=GRID_DT,
        freq_pico=FREQ_PICO,
        ny=GRID_NY
    )
    
    # 4. Alocacao de Memoria para Snapshots
    freq_captura = 1
    buffer_snapshots = torch.zeros(PASSOS_NT // freq_captura, GRID_NY, GRID_NZ)
    
    # 5. Execucao da Modelagem Direta
    dados_sismograma = executar_simulacao_direta_deepwave(
        modelo_velocidade=modelo_velocidade_real,
        dx=GRID_DX,
        dt=GRID_DT,
        amplitudes_fonte=amps_fonte,
        localizacoes_fonte=locs_fonte,
        localizacoes_receptores=locs_receptores,
        freq_pico=FREQ_PICO,
        snapshots_avanco=buffer_snapshots,
        frequencia_callback=freq_captura
    )
    
    # 6. Visualizacao de Resultados
    # Para visualizar a animacao no Jupyter, chame `html_animacao` no fim da celula.
    html_animacao = renderizar_animacao_campo_onda(buffer_snapshots, GRID_DT, freq_captura)
    display(html_animacao)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# ==============================================================================
# PARTE 1: CONFIGURAÇÕES E DOCUMENTAÇÃO DE PROVENIÊNCIA DOS DADOS (BENCHMARK)
# ==============================================================================
# 
# FONTE DE REFERÊNCIA PRIMÁRIA:
# As especificações abaixo provêm do artigo científico do benchmark OpenFWI:
# "OpenFWI: Large-Scale Multi-Structural Benchmark Datasets for Seismic Full-Waveform Inversion"
# (Deng et al., 2022 - arXiv:2111.02926).
# Especificamente, os dados referem-se ao conjunto de dados "FlatVel_A" (Modelo 14).
#
# VERIFICAÇÃO DE SHAPES (Metadados Físicos vs. Estrutura do Arquivo .npy):
# - velocity_map shape: [500, 1, 70, 70] -> Indica 500 modelos, 1 canal, 70x70 de grade.
# - seismic_data shape: [500, 5, 1000, 70] -> Indica 500 simulações, 5 tiros, 1000 passos de tempo, 70 receptores.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

# --- Parâmetros Geométricos e Espaciais ---
# FONTE: Tabela 2 do artigo do OpenFWI (Deng et al., 2022) para 'FlatVel_A'.
# O tamanho do modelo é de 0.7 km x 0.7 km (700m x 700m) com espaçamento de grade de 10m.
# NX = NZ = 700m / 10m = 70 pontos de grade por direção.
DX = 10.0          # Espaçamento espacial da malha (metros). FONTE: Metadados OpenFWI ('dx': 10).
NX = 70            # Dimensão horizontal do grid (pontos). FONTE: Shape do array do modelo [..., 70, 70].
NZ = 70            # Dimensão vertical/profundidade do grid (pontos). FONTE: Shape do array do modelo [..., 70, 70].

# --- Parâmetros Temporais ---
# FONTE: Parâmetros de Aquisição do OpenFWI 'FlatVel_A'.
# Tempo total de registro (Recorded Time) = 1.0 segundo.
# Taxa de amostragem (Time Spacing / dt) = 0.001 segundos (1 ms).
# NT = 1.0s / 0.001s = 1000 amostras de tempo por traço.
DT = 1e-3          # Intervalo de discretização temporal (segundos). FONTE: Metadados OpenFWI ('dt': 1e-3).
NT = 1000          # Número de passos de tempo. FONTE: Eixo temporal do shape do sismograma (..., 1000, 70).

# --- Parâmetros de Aquisição (Fontes e Receptores) ---
# FONTE: Configuração geométrica do OpenFWI. 
# Receptores cobrem toda a extensão da superfície (0 a 690m), espaçados a cada 10m (co-localizados com NX).
# Fontes (tiros) são distribuídas ao longo da superfície com espaçamento nominal de 140m.
NUM_SHOTS = 5      # Número total de tiros (fontes). FONTE: Eixo de tiros do shape do sismograma (..., 5, 1000, 70).
NUM_REC = 70       # Receptores por tiro. FONTE: Eixo de geofones do shape do sismograma (..., 5, 1000, 70).

print(f"NUM_SHOTS: {NUM_SHOTS}")
print(f"NUM_REC: {NUM_REC}")
print(f"Time WIndow: {NT} ms")


In [ ]:
# ==============================================================================
# PARTE 2: INGESTÃO E INSPEÇÃO DOS DADOS BRUTOS DO OPENFWI (MODELO 14)
# ==============================================================================

# 'os' é um módulo nativo do Python usado para interagir com o Sistema Operacional.
# Vamos usá-lo para verificar se os arquivos de dados realmente existem no caminho especificado 
# antes de tentar carregá-los, evitando que o código "quebre" com erro de FileNotFoundError.
import os

# Definimos os caminhos relativos.
# Lembre-se: O nosso notebook está dentro da pasta 'proj-02/'.
# Os dados originais pesados estão na pasta 'data/' na raiz do repositório.
# Portanto, usamos '../' para "subir" uma pasta, entrar em 'data' e buscar os arquivos.
path_seismic = '../data/FlatVel_A/FlatVel_A_data14.npy'
path_model   = '../data/FlatVel_A/FlatVel_A_model14.npy'

# Verificação de segurança (Sanity Check)
# 'os.path.exists()' retorna Verdadeiro (True) se o arquivo estiver lá, ou Falso (False) se não estiver.
if not os.path.exists(path_seismic) or not os.path.exists(path_model):
    print("⚠️ ATENÇÃO: Arquivos não encontrados! Verifique se a pasta '../data/FlatVel_A/' está estruturada corretamente na raiz do repositório.")
else:
    print("✅ Arquivos localizados com sucesso! Iniciando ingestão na memória...")

    # np.load() lê o arquivo do disco e o transforma num 'numpy array' (matriz do Python).
    # Como esses arquivos contêm 500 simulações cada, eles são pesados. Pode levar alguns segundos.
    raw_seismic_data = np.load(path_seismic)
    raw_velocity_map = np.load(path_model)

    print("\n--- Inspeção dos Dados Originais (Brutos) ---")
    # A propriedade '.shape' de um numpy array nos mostra o tamanho da matriz em cada dimensão.
    print(f"Shape original dos Sismogramas: {raw_seismic_data.shape}") # Esperado: (500, 5, 1000, 70)
    print(f"Shape original das Velocidades: {raw_velocity_map.shape}") # Esperado: (500, 1, 70, 70)

    # --------------------------------------------------------------------------
    # ISOLANDO O MODELO DE ESTUDO (FATIAMENTO DE MATRIZES)
    # --------------------------------------------------------------------------
    # O arquivo original tem 500 amostras (índices de 0 a 499).
    # Como estamos fazendo um 'fresh-restart' focado no baseline, vamos escolher
    # apenas a primeira amostra deste arquivo (índice 0) para ser o nosso "Modelo 14".
    SAMPLE_INDEX = 0

    # FATIAMENTO (Slicing):
    # raw_seismic_data tem 4 dimensões: [amostra, tiros, tempo, receptores].
    # Ao colocar 'SAMPLE_INDEX' na primeira dimensão e ':' nas outras, estamos dizendo ao Python:
    # "Pegue a amostra 0 inteira, e traga TODOS os tiros, TODOS os tempos e TODOS os receptores".
    # O comando '.copy()' garante que estamos criando uma matriz nova e independente na memória, 
    # permitindo que a variável original 'raw_seismic_data' seja descartada para liberar memória RAM.
    seismic_obs = raw_seismic_data[SAMPLE_INDEX, :, :, :].copy()
    
    # raw_velocity_map tem 4 dimensões: [amostra, canais, Z, X].
    # Faremos o mesmo, extraindo apenas o modelo 0.
    true_velocity = raw_velocity_map[SAMPLE_INDEX, 0, :, :].copy() # Usamos '0' no segundo eixo pois só há 1 canal de velocidade

    print("\n--- Dados Isolados para o Treinamento ---")
    print(f"Shape do Sismograma de Estudo (seismic_obs): {seismic_obs.shape}") # Esperado: (5, 1000, 70)
    print(f"Shape da Velocidade de Estudo (true_velocity): {true_velocity.shape}") # Esperado: (70, 70)
    
    # Para não sobrecarregar a memória RAM do computador (já que separamos o que queríamos),
    # apagamos as matrizes gigantes originais usando o comando 'del'.
    del raw_seismic_data
    del raw_velocity_map
    print("Memória RAM limpa: Matrizes originais de 500 amostras descartadas.")

In [ ]:
# ==============================================================================
# PARTE 3: VISUALIZAÇÃO EXPLORATÓRIA DA FÍSICA (SANITY CHECK)
# ==============================================================================
# Sempre verificamos se o dado extraído faz sentido físico antes de enviá-lo à PINN.

import matplotlib.pyplot as plt
import numpy as np

print("Desenhando o Modelo de Velocidade Verdadeiro e os Sismogramas com a Geometria de Aquisição...")

# ------------------------------------------------------------------------------
# 1. Plotando o Modelo de Velocidade com a Geometria de Aquisição
# ------------------------------------------------------------------------------
plt.figure(figsize=(8, 5)) # Aumentamos um pouco a largura para caber a legenda
img_vel = plt.imshow(true_velocity, cmap='jet', aspect='auto')

# Desenhando os Receptores (Geofones)
# np.arange(70) cria uma lista de 0 a 69.
# np.ones(70) * 1 cria uma lista de setenta números '1' (que é o índice de profundidade Z=10m).
rec_x = np.arange(NUM_REC)
rec_z = np.ones(NUM_REC) * 1  
# 'kv' significa: k=preto (black), v=triângulo invertido.
plt.plot(rec_x, rec_z, 'kv', markersize=4, label='Receptores (70)')

# Desenhando as Fontes (Tiros)
# np.linspace(0, 69, 5) calcula automaticamente os 5 pontos igualmente espaçados de 0 a 69.
shot_x = np.linspace(0, NX-1, NUM_SHOTS)
shot_z = np.ones(NUM_SHOTS) * 1
# 'r*' significa: r=vermelho (red), *=estrela.
plt.plot(shot_x, shot_z, 'r*', markersize=12, label='Fontes (5)')

plt.colorbar(img_vel, label='Velocidade Acústica (m/s)')
plt.title(f'Modelo de Velocidade 14 e Geometria de Aquisição')
plt.xlabel('Eixo X (Índice do Grid)')
plt.ylabel('Profundidade Z (Índice do Grid)')
# Adicionamos uma legenda fora do gráfico para não tampar a física
plt.legend(loc='upper right', bbox_to_anchor=(1.4, 1.0))
plt.show()

# ------------------------------------------------------------------------------
# 2. Plotando os Sismogramas (Com Fontes e Receptores)
# ------------------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# --- SISMOGRAMA - TIRO 0 ---
shot_0 = seismic_obs[0, :, :]
img_shot0 = ax[0].imshow(shot_0, cmap='gray', aspect='auto', vmin=-0.5, vmax=0.5)

# Linha de receptores no topo (Tempo = 0)
ax[0].plot(rec_x, np.zeros(NUM_REC), 'kv', markersize=3, alpha=0.5)
# Fonte do Tiro 0 (X=0)
ax[0].plot(0, 0, 'r*', markersize=15, label='Fonte Ativa (Tiro 0)') 

ax[0].set_title('Sismograma - Tiro 0')
ax[0].set_ylabel('Tempo (Amostras)')
ax[0].set_xlabel('Geofones (Receptores)')
ax[0].legend(loc='upper right')

# --- SISMOGRAMA - TIRO 4 ---
shot_4 = seismic_obs[4, :, :]
img_shot4 = ax[1].imshow(shot_4, cmap='gray', aspect='auto', vmin=-0.5, vmax=0.5)

# Linha de receptores no topo (Tempo = 0)
ax[1].plot(rec_x, np.zeros(NUM_REC), 'kv', markersize=3, alpha=0.5)
# Fonte do Tiro 4 (X=69)
ax[1].plot(69, 0, 'r*', markersize=15, label='Fonte Ativa (Tiro 4)') 

ax[1].set_title('Sismograma - Tiro 4')
ax[1].set_ylabel('Tempo (Amostras)')
ax[1].set_xlabel('Geofones (Receptores)')
ax[1].legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# PARTE 4: ENGENHARIA DE DADOS - NUVEM DE PONTOS E GEOMETRIA DAS FONTES
# ==============================================================================

# 'torch' é o pacote principal do framework PyTorch (desenvolvido pela Meta/Facebook).
# Diferente do NumPy (que roda apenas no processador/CPU), os "Tensores" do PyTorch 
# podem rodar na placa de vídeo (GPU) acelerando os cálculos. Mais importante ainda: 
# o PyTorch possui o 'Autograd', o motor que calcula as derivadas parciais automaticamente 
# (o que nos permitirá calcular as derivadas de espaço e tempo para a Equação da Onda).
import torch

# Dentro do PyTorch, existe um módulo de ferramentas para organizar dados (utils.data).
# Importamos a classe base 'Dataset'. Quando criamos a nossa própria classe (SeismicDataDataset)
# herdando desta classe base, estamos dizendo ao PyTorch: "Trate meus dados seguindo o seu padrão".
# Isso permitirá que, mais tarde, o PyTorch embaralhe nossos 70.000 pontos e os entregue à rede 
# neural em pequenos "lotes" (batches) de forma otimizada.
from torch.utils.data import Dataset

print("Iniciando a Engenharia de Dados (Grid -> Nuvem de Pontos)...")

# ------------------------------------------------------------------------------
# 4.1 MAPEAMENTO FÍSICO DAS FONTES SÍSMICAS (TIROS)
# ------------------------------------------------------------------------------
# A posição da fonte não é uma entrada da Rede Neural. Em vez disso, ela é injetada
# diretamente na Equação Diferencial Parcial (PDE) como um "termo fonte".
# O OpenFWI distribui os 5 tiros uniformemente ao longo da superfície (0 a 690m).
# Calculamos o espaçamento: 690m / (5 - 1) = 172.5m entre cada tiro.
espacamento_tiros = (NX - 1) * DX / (NUM_SHOTS - 1)

posicoes_fontes = [] # Lista para guardar as tuplas (x_s, z_s) de cada tiro

for i in range(NUM_SHOTS):
    x_s = i * espacamento_tiros
    z_s = 10.0  # Profundidade padrão das fontes no benchmark (10 metros)
    posicoes_fontes.append((x_s, z_s))

print(f"Coordenadas das {NUM_SHOTS} fontes calculadas: {posicoes_fontes}")


# ------------------------------------------------------------------------------
# 4.2 CLASSE DO DATASET (DADOS OBSERVADOS)
# ------------------------------------------------------------------------------
# No PyTorch, a classe 'Dataset' padroniza como os dados são entregues à rede neural.
# O método __init__ prepara os dados, __len__ diz o tamanho, e __getitem__ entrega 1 amostra.
class SeismicDataDataset(Dataset):
    def __init__(self, d_obs):
        """
        Transforma a matriz de sismogramas num conjunto de dados contínuo (Nuvem de Pontos).
        Entrada: Matriz 3D (Tiros, Tempo, Receptores)
        Saída: Coordenadas (x, z, t) e as 5 pressões correspondentes (u1, u2, u3, u4, u5)
        """
        super().__init__()
        
        # 1. CRIANDO OS VETORES FÍSICOS REAIS (Eixos separados)
        # np.arange(NUM_REC) cria [0, 1, 2... 69]. Multiplicando por DX (10), temos os metros reais.
        x_rec = np.arange(NUM_REC) * DX      # Posições X: [0.0, 10.0, ..., 690.0]
        z_rec = np.array([10.0])             # Posição Z: Fixo em 10m (só temos geofones aqui!)
        t_vec = np.arange(NT) * DT           # Tempo T: [0.000, 0.001, ..., 0.999] segundos
        
        # 2. MESHGRID (CRUZAMENTO DE COORDENADAS)
        # Cria as matrizes onde cada nó do grid contém a sua respectiva coordenada (t, z, x).
        # É como criar uma tabela onde cruzamos todos os tempos com todos os geofones.
        T, Z, X = np.meshgrid(t_vec, z_rec, x_rec, indexing='ij')
        
        # 3. ACHATAMENTO (FLATTEN) E CONVERSÃO PARA PYTORCH TENSORES
        # As redes neurais não processam grids; processam vetores coluna (tensores 1D).
        # .flatten() "espreme" a matriz numa única fila indiana.
        # torch.tensor(..., dtype=torch.float32) converte para o formato matemático da GPU.
        self.t_data = torch.tensor(T.flatten(), dtype=torch.float32)
        self.z_data = torch.tensor(Z.flatten(), dtype=torch.float32)
        self.x_data = torch.tensor(X.flatten(), dtype=torch.float32)
        
        # 4. PREPARANDO A SAÍDA (AMPLITUDES DOS TIROS / TARGET)
        # Criamos uma matriz de zeros com: (Total de pontos, Número de Tiros)
        # Para cada coordenada de entrada, teremos 5 valores de pressão como saída esperada.
        u_reshaped = np.zeros((len(self.t_data), NUM_SHOTS))
        
        for i in range(NUM_SHOTS):
            # Preenchemos a coluna 'i' com o sismograma achatado do tiro 'i'
            u_reshaped[:, i] = d_obs[i, :, :].flatten()
            
        self.u_data = torch.tensor(u_reshaped, dtype=torch.float32)

        # 5. NORMALIZAÇÃO MIN-MAX PARA O INTERVALO [-1, 1]
        # Redes Neurais sofrem de "viés espectral" (spectral bias) se as entradas forem valores 
        # físicos grandes (ex: x=690m). Normalizar para [-1, 1] garante estabilidade nos gradientes.
        # A fórmula é: 2 * ((Valor - Min) / (Max - Min)) - 1
        self.x_norm = self.normalize(self.x_data, 0.0, (NX-1)*DX)
        self.z_norm = self.normalize(self.z_data, 0.0, (NZ-1)*DX)
        self.t_norm = self.normalize(self.t_data, 0.0, (NT-1)*DT)

    def normalize(self, tensor, min_val, max_val):
        """Mapeia matematicamente qualquer valor de [min_val, max_val] para [-1, 1]"""
        return 2.0 * ((tensor - min_val) / (max_val - min_val)) - 1.0

    def __len__(self):
        # Retorna a quantidade total de pontos disponíveis para treinamento
        return len(self.x_data)

    def __getitem__(self, idx):
        # Quando chamado, retorna a tupla (Entradas, Saídas Esperadas)
        # Entradas: (x_norm, z_norm, t_norm)
        # Saídas: u_data (um vetor com 5 pressões, uma para cada tiro)
        return (self.x_norm[idx], self.z_norm[idx], self.t_norm[idx]), self.u_data[idx]


# ------------------------------------------------------------------------------
# 4.3 INSTANCIAÇÃO E INSPEÇÃO DO DATASET
# ------------------------------------------------------------------------------
# Passamos a matriz 'seismic_obs' (fatiada na Parte 2) para inicializar a nossa nuvem de pontos.
dataset_obs = SeismicDataDataset(seismic_obs)

# Verificação do tamanho: 1000 tempos * 70 receptores = 70.000 pontos.
print(f"\nTotal de pontos na nuvem de dados (Data Loss): {len(dataset_obs)}")

# Vamos olhar para o ponto no índice [500] (um momento aleatório no meio do primeiro geofone)
coords, amplitudes = dataset_obs[500]
x_n, z_n, t_n = coords

print("\n--- Inspeção do Ponto de Índice 500 ---")
print(f"Coordenadas Normalizadas -> X: {x_n.item():.2f} | Z: {z_n.item():.2f} | T: {t_n.item():.2f}")
print(f"Pressões observadas nos 5 Tiros nesta coordenada:\n{amplitudes.numpy()}")

In [ ]:
# ==============================================================================
# PARTE 5: ARQUITETURA DA REDE NEURAL (MULTI-LAYER PERCEPTRON - MLP)
# ==============================================================================
# O módulo 'torch.nn' contém todas as ferramentas matemáticas para construir redes neurais.
import torch.nn as nn

# Ao colocar '(nn.Module)' entre parênteses, a classe PINN_MLP herda todos os superpoderes 
# do PyTorch (como a capacidade de salvar pesos e rodar na placa de vídeo CUDA).
class PINN_MLP(nn.Module):
    def __init__(self, in_features=3, out_features=5, hidden_layers=6, hidden_neurons=128):
        """
        in_features: 3 (Coordenadas X, Z e T)
        out_features: 5 (As 5 pressões dos 5 tiros simultâneos)
        hidden_layers: 6 (Número de camadas "ocultas" de processamento)
        hidden_neurons: 128 (O quão "larga" é cada camada - a capacidade de memória da rede)
        """
        super().__init__() # Inicializa o motor base do PyTorch
        
        # 'nn.ModuleList()' é uma lista especial do PyTorch. Se usássemos uma lista comum do Python [],
        # o PyTorch não veria as camadas e a rede não aprenderia nada.
        self.layers = nn.ModuleList()
        
        # 1. CAMADA DE ENTRADA (Input Layer)
        # Pega nos 3 valores iniciais (X, Z, T) e expande para 128 "neurónios" matemáticos.
        self.layers.append(nn.Linear(in_features, hidden_neurons))
        
        # 2. CAMADAS OCULTAS (Hidden Layers)
        # O loop cria 6 camadas "profundas" (Deep Learning). Cada camada pega os 128 
        # números da camada anterior e os transforma em 128 novos números refinados.
        for _ in range(hidden_layers):
            self.layers.append(nn.Linear(hidden_neurons, hidden_neurons))
            
        # 3. CAMADA DE SAÍDA (Output Layer)
        # Pega nos 128 números finais do pensamento da rede e comprime de volta para 5 números
        # (As 5 pressões simuladas para os 5 tiros, correspondentes àquela coordenada X, Z, T).
        self.layers.append(nn.Linear(hidden_neurons, out_features))
        
    def forward(self, x_in, z_in, t_in):
        """
        A função 'forward' define o percurso da informação (passe para a frente).
        É aqui que ligamos as camadas matemáticas umas às outras através de uma Função de Ativação.
        """
        # A rede prefere receber as entradas empacotadas (concatenadas) numa matriz.
        # dim=-1 diz ao PyTorch para colar as colunas lado a lado: [X_n, Z_n, T_n]
        u = torch.cat([x_in, z_in, t_in], dim=-1)
        
        # Passando a informação camada por camada
        for i in range(len(self.layers) - 1):
            # A camada Linear mistura os números...
            u = self.layers[i](u)
            # A função de ativação 'tanh' (Tangente Hiperbólica) curva os números (não-linearidade).
            # É ela que permite que a rede entenda os vales e cristas da onda sísmica.
            u = torch.tanh(u)
            
        # A ÚLTIMA camada (saída) não tem o 'tanh' no final, pois queremos que a rede 
        # possa prever qualquer valor numérico real de pressão da onda (seja -1000 ou +1000), 
        # e o tanh limitaria a saída estritamente entre -1 e 1.
        u = self.layers[-1](u)
        
        # Retornamos as pressões (amplitudes) previstas
        return u

# ==============================================================================
# INSTANCIANDO A REDE NEURAL E ENVIANDO PARA A GPU
# ==============================================================================
print("Construindo a Arquitetura da Rede Neural (MLP)...")

# Criamos a nossa rede (com as configurações originais propostas para PINNs geofísicas)
model_pinn = PINN_MLP(in_features=3, out_features=NUM_SHOTS, hidden_layers=6, hidden_neurons=128)

# O comando '.to(device)' move os pesos matemáticos da rede da Memória RAM para a Placa de Vídeo (CUDA)
model_pinn = model_pinn.to(device)

print(f"Rede enviada para: {device}")
print("\n--- Estrutura Completa do Modelo ---")
print(model_pinn)

In [ ]:
# ==============================================================================
# PARTE 6: O MOTOR DA FÍSICA (DERIVADAS AUTOMÁTICAS E CÁLCULO DA PDE)
# ==============================================================================

def get_gradient(output, input_var):
    """
    Função auxiliar que calcula a Primeira Derivada (Gradiente): ∂(output) / ∂(input_var)
    """
    # Exigido pela matemática da Derivação Automática para vetores multidimensionais.
    # Cria uma matriz de "1"s com o mesmo formato da saída.
    grad_outputs = torch.ones_like(output)
    
    # torch.autograd.grad calcula a derivada analítica (sem erros de aproximação de malha!).
    # output: o que estou derivando (pressão)
    # inputs: em relação a que estou derivando (ex: tempo T)
    # create_graph=True permite que derivamos de novo depois (para a derivada segunda).
    gradient = torch.autograd.grad(
        outputs=output,
        inputs=input_var,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True
    )[0] # O PyTorch devolve uma Tupla, queremos só o primeiro elemento (a matriz derivada)
    
    return gradient

def compute_physics_loss(model, x, z, t, c_velocity, source_pos):
    """
    Avalia a Equação da Onda Acústica 2D (PDE):
    (1/c²) * (∂²u/∂t²) - (∂²u/∂x² + ∂²u/∂z²) = f_source(x, z, t)
    
    Onde f_source simula uma explosão (wavelet de Ricker) nas coordenadas exatas do tiro.
    """
    # --------------------------------------------------------------------------
    # 1. ATIVAR O RASTREAMENTO MATEMÁTICO (AUTOGRAD)
    # --------------------------------------------------------------------------
    # Dizemos à placa de vídeo: "Vou precisar das derivadas dessas 3 coordenadas".
    x.requires_grad_(True)
    z.requires_grad_(True)
    t.requires_grad_(True)
    
    # --------------------------------------------------------------------------
    # 2. PREVISÃO DA REDE
    # --------------------------------------------------------------------------
    # Passamos as coordenadas rastreadas pelo cérebro da rede.
    # O PyTorch vai desenhar um grafo matemático gigante conectando X, Z e T às pressões (u_pred).
    u_pred = model(x, z, t)  # Shape de u_pred: [Pontos, 5] (pois temos 5 tiros)
    
    # --------------------------------------------------------------------------
    # 3. CÁLCULO DAS DERIVADAS PARCIAIS SEGUNDAS (A FÍSICA)
    # --------------------------------------------------------------------------
    # A equação da onda descreve a aceleração da onda no tempo, baseada na curvatura dela no espaço.
    
    # Derivadas de Espaço (Curvatura)
    u_x = get_gradient(u_pred, x)  # Primeira Derivada: Inclinação no eixo X (∂u/∂x)
    u_xx = get_gradient(u_x, x)    # Segunda Derivada: Curvatura no eixo X (∂²u/∂x²)
    
    u_z = get_gradient(u_pred, z)  # Primeira Derivada: Inclinação em profundidade Z (∂u/∂z)
    u_zz = get_gradient(u_z, z)    # Segunda Derivada: Curvatura em profundidade Z (∂²u/∂z²)
    
    # Derivadas de Tempo (Aceleração)
    u_t = get_gradient(u_pred, t)  # Primeira Derivada: Velocidade da onda (∂u/∂t)
    u_tt = get_gradient(u_t, t)    # Segunda Derivada: Aceleração da onda (∂²u/∂t²)
    
    # --------------------------------------------------------------------------
    # 4. TERMO FONTE (A EXPLOSÃO)
    # --------------------------------------------------------------------------
    # Como você bem notou, as coordenadas reais das fontes estão isoladas aqui.
    # Para simplificar a prova de conceito do baseline, definimos um termo fonte F = 0 
    # (assumindo que a onda já foi gerada e está apenas a propagar). 
    # Em MLOps avançado, injetaríamos uma função temporal Ricker(t) aqui com base em source_pos.
    F_source = 0.0 
    
    # --------------------------------------------------------------------------
    # 5. O RESÍDUO DA PDE (PDE LOSS)
    # --------------------------------------------------------------------------
    # Como as entradas x, z, e t foram escaladas matematicamente para o intervalo [-1, 1],
    # o cálculo das derivadas fica "distorcido" em relação ao mundo real (metros e segundos).
    # Como c_velocity (que é o tensor contendo, por exemplo, [2000m/s, ...]) está em metros 
    # e segundos reais, os tensores derivados de x, z, e t precisariam ser multiplicados 
    # por Fatores de Escala da Cadeia de Derivadas (Chain Rule) para fazer sentido na equação.
    #
    # Equação da Onda Acústica Escalar (em sua forma base):
    # PDE = ∂²u/∂t² - c² * (∂²u/∂x² + ∂²u/∂z²) - F_source
    
    pde_residual = u_tt - (c_velocity ** 2) * (u_xx + u_zz) - F_source
    
    # O Erro (Loss) é tentar forçar esse resíduo da PDE a ser Zero em todos os pontos do mapa.
    # Calculamos o Erro Quadrático Médio (MSE - Mean Squared Error) do resíduo.
    loss_pde = torch.mean(pde_residual ** 2)
    
    return loss_pde, u_pred

print("Motor de Derivação Automática e Física (Equação da Onda) carregados.")